## Directions

1. Create a python file that webscrapes GDP by country.https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)  and plots a stacked interactive bar plot using plotly. Stack countries within regions using the IMF numbers. Fill in the missing data with predictions from the UN and Worldbank. Please include this in your ipython notebook and output your plot to an html file containing the plot. The region data can be found from an older table here https://en.wikipedia.org/w/index.php?title=List_of_countries_by_GDP_(nominal)&oldid=1187446467.
2. Look at the chapter on interactive graphics https://smart-stats.github.io/ds4bio_book/interactive/ and, specifically, the code to display a subject's MRICloud data as a sunburst plot. Do the following. Display this subject's data as a Sankey diagram. https://plotly.com/python/sankey-diagram/ Display as many levels as you can (at least 3) for Type = 1, starting from the intracranial volume.
3. Create a simple webpage containing the Sankey graphic and host it on github pages. You'll have to create a new public repo from your regular github account and add this file. Put the link to your live web page in a markdown cell of your hw5.ipynb file as a text block.

Your homework should include

An file called hw5.ipynb that has your code for parts 1 and 2.
Two html files, one called sankey.html and one called stacked_bar.html that contain the two plots as html files.
Your hw5.ipynb file should have a text block that contains a link to the live and publiclyl hosted sankey diagram.
Remember that you should have two repositories for this assignment. First, you need the HW repository that you create when you accept the assignment. This is where you do the work for the three files (hw5.ipynb, sankey.html, stacked_bar.html). Secondly, you will need to create your own repository containing a live link to your sankey html file. So, when I click on that link it should show a page containing your plot.

Note plotly objects contain a method called to_html() which is useful for creating an html file.



## Question 1

In [10]:
#setup

%pip install lxml
%pip install requests
%pip install beautifulsoup4

import requests as rq
import bs4
import pandas as pd
import plotly.express as px
import numpy as np

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [11]:
#read URL
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)'
page = rq.get(url, headers={"User-Agent": "Chrome"})
## print out the first 200 characters just to see what it looks like
page.text[0 : 99]

#read the page into bs4.
bs4page = bs4.BeautifulSoup(page.text, 'html.parser')
tables = bs4page.find_all('table',{'class':"wikitable"})

#read html into pandas
from io import StringIO
# Read the table from the StringIO object into pandas
# Note most recent version of pandas won't accept a string as input, it needs to be passed through stringio
countries = pd.read_html(StringIO(str(tables[0])))[0]
countries.head()

countries.columns
#renaming
countries.columns = [
    'country',
    'imf',
    'world_bank',
    'un'
]


In [12]:
#read secondary url
secondary_url = 'https://en.wikipedia.org/w/index.php?title=List_of_countries_by_GDP_(nominal)&oldid=1187446467'
secondary_page = rq.get(secondary_url, headers={"User-Agent": "Chrome"})

#parse
secondary_bs4page = bs4.BeautifulSoup(
    secondary_page.text,
    'html.parser'
)

secondary_tables = secondary_bs4page.find_all('table')

len(secondary_tables)

#read html into pandas
old_tables = pd.read_html(StringIO(str(secondary_bs4page)))

old_countries = old_tables[2]
old_countries.head()

old_countries.columns

#renaming columns 
old_countries.columns = [
    'country',
    'region',
    'imf',
    'imf_year',
    'world_bank',
    'world_bank_year',
    'un',
    'un_year'
]

# merging
regions = old_countries[['country', 'region']]

countries = countries.merge(
    regions,
    on='country',
    how='left'
)

countries.head()


,country,imf,world_bank,un,region
0,World,126295331,118350166,111565144,—
1,United States,32383920,30769700,29298000,Americas
2,China[n 1],20851593,19498039,18743802,NaN
3,Germany,5452858,5050923,4659929,Europe
4,Japan,4379253,4435163,4026211,Asia


In [13]:
#need numeric data
for col in ['imf', 'world_bank', 'un']:
    countries[col] = pd.to_numeric(countries[col], errors='coerce')

#estimating missing values
countries['gdp'] = (
    countries['imf']
    .fillna(countries['world_bank'])
    .fillna(countries['un'])
)
#fixing some misnamed region
countries['country'] = countries['country'].str.replace(r'\[.*?\]', '', regex=True)
old_countries['country'] = old_countries['country'].str.replace(r'\[.*?\]', '', regex=True)

countries = countries.drop(columns='region')

regions = old_countries[['country', 'region']]

countries = countries.merge(
    regions,
    on='country',
    how='left'
)

countries = countries[countries['country'] != 'World']

#checking for countries still missing a region
countries[countries['region'].isna()]

#dropping rows missing either gdp or region:
plot_data = countries.dropna(subset=['gdp', 'region']).copy()

plot_data.head()

,country,imf,world_bank,un,gdp,region
1,United States,32383920.0,30769700.0,29298000.0,32383920.0,Americas
2,China,20851593.0,19498039.0,18743802.0,20851593.0,Asia
3,Germany,5452858.0,5050923.0,4659929.0,5452858.0,Europe
4,Japan,4379253.0,4435163.0,4026211.0,4379253.0,Asia
5,United Kingdom,4264794.0,4002588.0,3685881.0,4264794.0,Europe


In [14]:
#Plotting
fig = px.bar(
    plot_data,
    x='region',
    y='gdp',
    color='country',
    title='GDP by Region and Country',
    labels={
        'region': 'Region',
        'gdp': 'GDP',
        'country': 'Country'
    }
)

fig.update_layout(
    barmode='stack',
    yaxis_title='GDP (millions of US dollars)'
)

fig.show()

In [15]:
import os
os.getcwd()

fig.write_html(
    "/Users/elisabethlaviolette/Documents/GitHub/HW5/stacked_bar.html",
    include_plotlyjs="cdn"
)

## Question 2


In [16]:
# load in the hierarchy information
url = "https://raw.githubusercontent.com/bcaffo/MRIcloudT1volumetrics/master/inst/extdata/multilevel_lookup_table.txt"
multilevel_lookup = pd.read_csv(url, sep = "\t").drop(['Level5'], axis = 1)
multilevel_lookup = multilevel_lookup.rename(columns = {
    "modify"   : "roi",
    "modify.1" : "level4",
    "modify.2" : "level3",
    "modify.3" : "level2",
    "modify.4" : "level1"})
multilevel_lookup = multilevel_lookup[['roi', 'level4', 'level3', 'level2', 'level1']]
multilevel_lookup.head()


##load in the subject data
id = 127
subjectData = pd.read_csv("https://raw.githubusercontent.com/smart-stats/ds4bio_book/main/book/assetts/kirby21AllLevels.csv")
subjectData = subjectData.loc[(subjectData.type == 1) & (subjectData.level == 5) & (subjectData.id == id)]
subjectData = subjectData[['roi', 'volume']]
## Merge the subject data with the multilevel data
subjectData = pd.merge(subjectData, multilevel_lookup, on = "roi")
subjectData = subjectData.assign(icv = "ICV")
subjectData = subjectData.assign(comp = subjectData.volume / np.sum(subjectData.volume))
subjectData.head()






,roi,volume,level4,level3,level2,level1,icv,comp
0,SFG_L,12926,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.009350
1,SFG_R,10050,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R,ICV,0.007270
2,SFG_PFC_L,12783,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.009247
3,SFG_PFC_R,11507,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R,ICV,0.008324
4,SFG_pole_L,3078,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.002227


In [17]:
import plotly.graph_objects as go
import pandas as pd

# Build links for ICV: level1
links1 = (
    subjectData.groupby("level1", as_index=False)["volume"]
    .sum()
    .assign(source="ICV")
    .rename(columns={"level1": "target"})
)

# Build links for level1 -> level2
links2 = (
    subjectData.groupby(["level1", "level2"], as_index=False)["volume"]
    .sum()
    .rename(columns={"level1": "source", "level2": "target"})
)

# Build links for level2 -> level3
links3 = (
    subjectData.groupby(["level2", "level3"], as_index=False)["volume"]
    .sum()
    .rename(columns={"level2": "source", "level3": "target"})
)

# level3 -> level4
links4 = (
    subjectData.groupby(["level3", "level4"], as_index=False)["volume"]
    .sum()
    .rename(columns={"level3": "source", "level4": "target"})
)
# level4 -> roi
links5 = (
    subjectData.groupby(["level4", "roi"], as_index=False)["volume"]
    .sum()
    .rename(columns={"level4": "source", "roi": "target"})
)
# Combine all links
links = pd.concat(
    [links1, links2, links3, links4, links5],
    ignore_index=True
)
# Give nodes unique IDs by hierarchy level
links1["source_id"] = "ICV"
links1["target_id"] = "L1_" + links1["target"].astype(str)

links2["source_id"] = "L1_" + links2["source"].astype(str)
links2["target_id"] = "L2_" + links2["target"].astype(str)

links3["source_id"] = "L2_" + links3["source"].astype(str)
links3["target_id"] = "L3_" + links3["target"].astype(str)

links4["source_id"] = "L3_" + links4["source"].astype(str)
links4["target_id"] = "L4_" + links4["target"].astype(str)

# Combine links
links = pd.concat(
    [links1, links2, links3, links4],
    ignore_index=True
)

# Unique node IDs
node_ids = pd.unique(
    pd.concat(
        [links["source_id"], links["target_id"]],
        ignore_index=True
    )
).tolist()

# Labels shown on the graph
labels = [
    x.replace("L1_", "")
     .replace("L2_", "")
     .replace("L3_", "")
     .replace("L4_", "")
    for x in node_ids
]

labels = pd.unique(
    pd.concat([links["source"], links["target"]], ignore_index=True)
).tolist()

label_to_index = {label: i for i, label in enumerate(labels)}

source = links["source"].map(label_to_index)
target = links["target"].map(label_to_index)
value = links["volume"]

# Create Sankey diagram
fig = go.Figure(
    data=[
        go.Sankey(
            node=dict(
                pad=5,
                thickness=8,
                line=dict(color="black", width=0.5),
                label=labels
            ),
            link=dict(
                source=source,
                target=target,
                value=value
            )
        )
    ]
)

fig.update_layout(
    title_text="MRICloud Brain Volume: Sankey Diagram",
    width=1600,
    height=800,
    font_size=8
)

fig.update_layout(
    title_text="MRICloud Brain Volume: Sankey Diagram",
    width=1600,
    height=900
)
fig.show()

fig.write_html(
    "/Users/elisabethlaviolette/Documents/GitHub/HW5/sankey.html",
    include_plotlyjs="cdn"
)

## Quesiton 3

[Click Here!](https://elisabethlavi.github.io/HW5/sankey.html)